# Documentation des données — PAYSIM

Ce document couvre : la recherche et l'évaluation des sources, le dictionnaire et la traçabilité des données.

Ce notebook complète [01_extraction_paysim.ipynb](../01_extraction_paysim.ipynb).

## Sommaire

1. Source principale : PaySim
2. Contrôles de qualité
3. Dictionnaire métier et disponibilité des variables
4. Risques et données personnelles
5. Source complémentaire et sources écartées
6. Traçabilité et conditions de réutilisation
7. Transmission à l’équipe


## 1. Source principale : PaySim

[PaySim sur Kaggle](https://www.kaggle.com/datasets/ealaxi/paysim1) est le jeu synthétique selectionné pour ce sujet. Une ligne représente une transaction de paiement mobile, la cible `isFraud` vaut 1 pour une fraude et 0 sinon.

L'objectif est de détecter une fraude lorsque la transaction est initiée, **avant son exécution**. Le CSV complet est réutilisé depuis le cache KaggleHub de l’extraction, ou depuis sa copie locale dans `docs/sources/` lorsqu’elle existe. L’empreinte du journal l’identifie.


### Évaluation selon les 7 critères

La source est évaluée selon :
la pertinence, la couverture, la fraîcheur, la granularité, la qualité,
biais de collecte et licence.

| Critère | Évaluation de PaySim |
|---|---|
| Pertinence | Transactions de paiement mobile avec cible de fraude, adaptées au sujet. |
| Couverture | 6 362 620 transactions et 11 variables ; `step` observé de 1 à 743. |
| Fraîcheur | Jeu statique de simulation, sans actualisation des comportements réels. |
| Granularité | Une ligne par transaction. |
| Qualité | Structure, valeurs manquantes, doublons et cible contrôlés ; conventions de soldes à interpréter. |
| Biais | Hypothèses du simulateur et classes très déséquilibrées. |
| Licence | CC BY-SA 4.0 consignée dans la documentation et le journal. |








#### • Pertinence
Le dataset est directement pertinent pour notre problématique de détection
de fraude aux paiements mobiles.

Les variables transactionnelles comme "type", "amount" et les soldes
fournissent des informations explicatives.

#### • Couverture
Les transactions du jeu de données appartiennent à cinq catégories :
- CASH_IN
- CASH_OUT
- DEBIT
- PAYMENT
- TRANSFER.


#### • Fraîcheur
PaySim est un dataset statique : généré par simulation. Il ne s'agit pas d'un flux de transactions actualisé en temps réel.
Il ne permet donc pas de représenter directement l'évolution récente des comportements frauduleux.

La variable "step" représente un pas temporel d'une heure. Le fichier contient 743 valeurs distinctes de "step" (de 1 à 743). Cela représente environ 31 jours de simulation (743 heures/24)


#### • Granularité
La granularité est adaptée à notre problématique :
**1 ligne = 1 transaction.**

Cette unité correspond directement à la décision que nous souhaitons
améliorer : déterminer si une transaction est frauduleuse.


#### • Qualité
Les contrôles initiaux réalisés dans le notebook [01_extraction_paysim.ipynb](../01_extraction_paysim.ipynb) montrent :

- 0 valeur manquante ;
- 0 ligne dupliquée ;
- une structure de 11 variables ;
- une cible fortement déséquilibrée.

La qualité structurelle du fichier est donc satisfaisante mais des précautions restent cependant nécessaires concernant la signification
métier de certaines variables et leur disponibilité au moment de la prédiction.


#### • Biais de collecte
PaySim est un jeu de données synthétique issu d'une simulation financière. Les comportements observés ne correspondent pas nécessairement
à ceux de vrais utilisateurs, cela dépend des hypothèses du simulateur.

Un modèle performant sur PaySim ne garantit pas les mêmes performances
sur des transactions réelles.


La documentation officielle sur Kaggle indique que la génération des données a été réalisée à partir de plusieurs exécutions
avec des graines aléatoires différentes, pour potentiellement vérifier que le résultat reste fiable.

Les résultats obtenus sur PaySim doivent donc être interprétés avec prudence
avant une intégration à un environnement bancaire réel.

#### • Licence
(Creative Commons Attribution-ShareAlike 4.0 International).

Cette licence autorise le partage et l'adaptation du contenu, y compris pour
un usage commercial, à condition de citer la source et de conserver une
licence compatible en cas de redistribution d'un contenu modifié.

### Initialisation des chemins du projet

Cette cellule initialise les bibliothèques utilisées dans le notebook et recherche automatiquement la racine du projet à partir du fichier `journal_lineage.csv`. Elle permet ensuite de construire le chemin vers le fichier PaySim sans utiliser de chemin absolu propre à un ordinateur, ce qui facilite l’exécution du notebook par les différents membres du groupe.


In [8]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

class StopExecution(Exception):
    """Arrêter la cellule après le message, sans traceback dans Jupyter."""
    def _render_traceback_(self):
        return []

racine = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "docs/journal_lineage.csv").is_file()), None)
if racine is None:
    print("Ouvrir le notebook depuis le dossier du projet ou docs/.")
    raise StopExecution

# Conserver la copie locale lorsqu’elle existe ; sinon réutiliser le cache de l’extraction.
source_paysim = Path("docs/sources/PS_20174392719_1491204439457_log-2.csv")
if not (racine / source_paysim).is_file():
    source_paysim = None
    try:
        from kagglehub.config import get_cache_folder
    except ImportError:
        pass
    else:
        versions = Path(get_cache_folder()) / "datasets/ealaxi/paysim1/versions"
        dossiers = sorted(
            (d for d in versions.iterdir() if d.is_dir() and d.name.isdigit()),
            key=lambda d: int(d.name), reverse=True
        ) if versions.is_dir() else []
        for dossier in dossiers:
            csv_files = sorted(dossier.rglob("*.csv"))
            if len(csv_files) == 1:
                source_paysim = Path(os.path.relpath(csv_files[0], racine))
                break
            if csv_files:
                print("Plusieurs CSV PaySim en cache : exécuter 01_extraction_paysim.ipynb pour sélectionner le fichier.")
                break

def fichier_paysim():
    """Résoudre le chemin relatif sans télécharger ni modifier la donnée."""
    if source_paysim is None or not (racine / source_paysim).is_file():
        print("PaySim est absent. Exécuter d’abord 01_extraction_paysim.ipynb pour télécharger le CSV, puis relancer la cellule d’initialisation de ce notebook.")
        raise StopExecution
    return racine / source_paysim

fichier_paysim()


## 2. Contrôles de qualité

Les valeurs ci-dessous reprennent les contrôles enregistrés sur le CSV complet et les résultats du notebook d’extraction. 

Aucun doublon n’a été détecté dans PaySim lors des contrôles réalisés. Un solde nul ne constitue pas une valeur manquante. Les écarts entre montant et solde ne suffisent pas à conclure à une erreur : ils dépendent des conventions du simulateur.

Les deux tableaux présentent la synthèse des contrôles et la répartition des transactions par type. Les contrôles enregistrés indiquent également 16 montants nuls et aucun montant négatif.


In [18]:
from pathlib import Path
from IPython.display import HTML, display

# Repérer le projet depuis la racine, docs/ ou un sous-dossier.
racine_projet = next((d for d in (Path.cwd(), *Path.cwd().parents)
                       if (d / "docs/journal_lineage.csv").is_file()
                       and (d / "docs/02_documentation_data.ipynb").is_file()), None)
if racine_projet is None:
    raise FileNotFoundError("Ouvrir le notebook depuis le dossier du projet ou l’un de ses sous-dossiers.")

def afficher(objet):
    """Afficher les tableaux sans aucune bordure, avec défilement si nécessaire."""
    if hasattr(objet, "to_html"):
        if hasattr(objet, "hide"):
            contenu = objet.to_html()
        else:
            contenu = objet.to_html(index=False, border=0)
    elif isinstance(objet, HTML):
        contenu = objet.data
    else:
        display(objet)
        return
    style = """<style>
    .table-donnees table {border-collapse:collapse !important; border:0 !important; font-size:13px;}
    .table-donnees table, .table-donnees thead, .table-donnees tbody, .table-donnees tr, .table-donnees th, .table-donnees td {border:0 !important; outline:0 !important; box-shadow:none !important; padding:8px 10px !important;
      text-align:left !important; vertical-align:top; line-height:1.45;}
    .table-donnees thead th {border-bottom:0 !important; font-weight:600;}
    .table-donnees tbody tr {background:transparent !important;}
    .table-donnees details {margin:8px 0;}
    </style>"""
    display(HTML(style + '<div class="table-donnees" style="max-width:100%;overflow-x:auto;">' + contenu + '</div>'))


In [21]:
from pathlib import Path
import pandas as pd

import hashlib
import numpy as np


def controler_paysim(fichier):
    """Contrôler le CSV par blocs, sans modifier les données sources."""
    fichier = Path(fichier)
    empreinte = hashlib.sha256()
    with fichier.open("rb") as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b""):
            empreinte.update(bloc)

    resultats = {
        "fichier": fichier.name, "sha256": empreinte.hexdigest(),
        "colonnes": pd.read_csv(fichier, nrows=0).columns.tolist(),
        "lignes": 0, "fraudes": 0, "valeurs_manquantes": 0,
        "montants_nuls": 0, "montants_negatifs": 0, "par_type": {}
    }
    cible_valeurs, steps, empreintes_lignes = set(), set(), []
    # Fixer les types pour comparer les empreintes entre tous les blocs.
    types = {"step": "int64", "isFraud": "int64", "isFlaggedFraud": "int64",
             "type": "object", "nameOrig": "object", "nameDest": "object",
             "amount": "float64", "oldbalanceOrg": "float64",
             "newbalanceOrig": "float64", "oldbalanceDest": "float64",
             "newbalanceDest": "float64"}
    for bloc in pd.read_csv(fichier, dtype=types, chunksize=250_000):
        resultats["lignes"] += len(bloc)
        resultats["fraudes"] += int(bloc["isFraud"].sum())
        resultats["valeurs_manquantes"] += int(bloc.isna().sum().sum())
        resultats["montants_nuls"] += int(bloc["amount"].eq(0).sum())
        resultats["montants_negatifs"] += int(bloc["amount"].lt(0).sum())
        cible_valeurs.update(bloc["isFraud"].unique().tolist())
        steps.update(bloc["step"].unique().tolist())
        empreintes_lignes.append(pd.util.hash_pandas_object(bloc, index=False).to_numpy())
        for type_transaction, groupe in bloc.groupby("type"):
            total = resultats["par_type"].setdefault(type_transaction,
                                                   {"transactions": 0, "fraudes": 0})
            total["transactions"] += len(groupe)
            total["fraudes"] += int(groupe["isFraud"].sum())

    hashes = np.concatenate(empreintes_lignes) if empreintes_lignes else np.array([], dtype="uint64")
    resultats["doublons_empreinte_ligne"] = len(hashes) - len(np.unique(hashes))
    resultats["cible_valeurs"] = sorted(cible_valeurs)
    resultats["step_min"] = min(steps) if steps else None
    resultats["step_max"] = max(steps) if steps else None
    resultats["step_distincts"] = len(steps)
    resultats["taux_fraude_pct"] = 100 * resultats["fraudes"] / resultats["lignes"] if resultats["lignes"] else 0
    resultats["par_type"] = dict(sorted(resultats["par_type"].items()))
    return resultats

fichier_paysim()
controles = controler_paysim(fichier_paysim())
resume_controles = pd.DataFrame([
    ("Fichier", controles["fichier"]), ("SHA-256", controles["sha256"]),
    ("Transactions", controles["lignes"]), ("Variables", len(controles["colonnes"])),
    ("Fraudes", controles["fraudes"]), ("Fraudes (%)", round(controles["taux_fraude_pct"], 4)),
    ("Valeurs manquantes", controles["valeurs_manquantes"]),
    ("Montants nuls (amount = 0)", controles["montants_nuls"]),
    ("Montants négatifs (amount < 0)", controles["montants_negatifs"]),
    ("Doublons par empreinte de ligne", controles["doublons_empreinte_ligne"]),
    ("Valeurs de la cible", str(controles["cible_valeurs"])),
    ("Pas temporels min / max / distincts", f'{controles["step_min"]} / {controles["step_max"]} / {controles["step_distincts"]}')
], columns=["Contrôle", "Résultat"])
afficher(resume_controles.style.hide(axis="index"))
repartition_types = pd.DataFrame.from_dict(controles["par_type"], orient="index").rename_axis("Type").reset_index()
repartition_types["Fraudes (%)"] = (100 * repartition_types.fraudes / repartition_types.transactions).round(4)
afficher(repartition_types.style.hide(axis="index"))


Contrôle,Résultat
Fichier,PS_20174392719_1491204439457_log-2.csv
SHA-256,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b
Transactions,6362620
Variables,11
Fraudes,8213
Fraudes (%),0.129100
Valeurs manquantes,0
Doublons par empreinte de ligne,0
Valeurs de la cible,"[0, 1]"
Pas temporels min / max / distincts,1 / 743 / 743


Type,transactions,fraudes,Fraudes (%)
CASH_IN,1399284,0,0.000000
CASH_OUT,2237500,4116,0.184000
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000
TRANSFER,532909,4097,0.768800


Les fraudes observées se répartissent entre `CASH_OUT` (4 116) et `TRANSFER` (4 097). Cette association appartient au jeu simulé et ne décrit pas toutes les fraudes réelles. Les distributions et la cohérence métier des soldes seront approfondies dans l’EDA.

Trois hypothèses pour l’EDA : 
- le type est associé au risque 
- les montants diffèrent selon la classe 
- la relation montant/solde avant transaction est associée à la fraude. Elles ne constituent pas des conclusions causales.

• Vérification que les variables du Dictionnaire correspondent aux colonnes du fichier :

In [24]:
# Lire uniquement l’en-tête pour vérifier les colonnes du fichier local.
colonnes = pd.read_csv(fichier_paysim(), nrows=0).columns.tolist()


## 3. Dictionnaire métier et disponibilité des variables

Le [dictionnaire](dictionnaire_variables.csv) distingue le type métier, le rôle, la disponibilité et la recommandation pour la suite.

`newbalanceOrig` et `newbalanceDest` sont exclues des variables prédictives car elles décrivent les soldes après transaction. `isFlaggedFraud` est la sortie d’une règle existante, pas une entrée métier indépendante.

**État de l’extraction :** dans [01_extraction_paysim.ipynb](01_extraction_paysim.ipynb) `df_model` retire ces trois colonnes et conserve les deux identifiants. 

**Recommandation future:** ne pas utiliser `nameOrig` et `nameDest` dans le modèle de référence, en raison de leur forte cardinalité. Cette recommandation n’est pas une transformation déjà appliquée.


In [25]:
dictionnaire = pd.read_csv(racine / "docs/dictionnaire_variables.csv", sep=";")
assert dictionnaire.variable.tolist() == colonnes
with pd.option_context("display.max_colwidth", 100):
    display(dictionnaire)


,variable,type_metier,description,role,disponibilite,decision,justification
0,step,Quantitative discrète,Pas temporel horaire de la simulation (valeurs observées : 1 à 743),Variable explicative,"Avant exécution, à confirmer en contexte réel",Candidate,Effet temporel à étudier lors de l’EDA.
1,type,Qualitative nominale,Type de transaction,Variable explicative,"Avant exécution, à confirmer en contexte réel",Candidate,Répartition des fraudes à étudier selon le type de transaction.
2,amount,Quantitative continue,Montant de la transaction,Variable explicative,"Avant exécution, à confirmer en contexte réel",Candidate,Distribution et valeurs extrêmes à étudier lors de l’EDA.
3,nameOrig,Qualitative nominale / identifiant,Identifiant du compte émetteur,Identifiant,Avant exécution,Utilisation brute non recommandée pour le modèle de référence,Identifiant à forte cardinalité ; conservé dans le df_model actuel.
4,oldbalanceOrg,Quantitative continue,Solde du compte émetteur avant la transaction,Variable explicative,"Avant exécution, à confirmer en contexte réel",Candidate,Cohérence avec le montant à vérifier.
5,newbalanceOrig,Quantitative continue,Solde du compte émetteur après la transaction,Information postérieure,Après exécution,Exclue,Solde après transaction : fuite temporelle.
6,nameDest,Qualitative nominale / identifiant,Identifiant du compte destinataire,Identifiant,Avant exécution,Utilisation brute non recommandée pour le modèle de référence,Identifiant à forte cardinalité ; conservé dans le df_model actuel.
7,oldbalanceDest,Quantitative continue,Solde du destinataire avant la transaction,Variable explicative,"Avant exécution, à confirmer en contexte réel",Candidate,Soldes nuls (0) à distinguer des valeurs manquantes.
8,newbalanceDest,Quantitative continue,Solde du destinataire après la transaction,Information postérieure,Après exécution,Exclue,Solde après transaction : fuite temporelle.
9,isFraud,Qualitative nominale binaire,Indique si la transaction est frauduleuse,Variable cible,Étiquette de supervision,Cible uniquement,Jamais une variable explicative ; classes très déséquilibrées.


## 4. Risques et données personnelles

Les données utilisées dans ce projet ne correspondent pas à des transactions bancaires réelles en libre accès. PaySim est un jeu de données synthétique. Dans le secteur financier, les transactions réelles sont rarement diffusées publiquement au niveau individuel en raison des contraintes de confidentialité, de sécurité et de protection des données. Les sources institutionnelles comme l’OSMP publient principalement des statistiques agrégées, utiles pour contextualiser le phénomène de fraude mais non pour entraîner directement un modèle transactionnel. Cela ne signifie pas qu’aucun jeu de données réel anonymisé n’existe.

- **Fuite de données (Data Leakage) :** les soldes après transaction ne sont pas disponibles à l’initiation. La sortie `isFlaggedFraud` doit rester séparée des variables explicatives.

- **Classes rares - déséquilibre:** 0,1291 % de fraudes ; l’accuracy seule peut masquer l’absence de détection.

- **Simulation :** une performance sur PaySim ne démontre pas une généralisation à un système réel. Une validation sur des transactions réelles serait nécessaire avant un usage réel en production.

PaySim est synthétique. À notre connaissance, la documentation disponible ne relie pas ses identifiants à des personnes physiques réelles ; le risque RGPD est donc limité. Cela ne constitue pas une preuve d’anonymisation. 

Dans un système réel, identifiants et historiques financiers seraient des données personnelles à protéger : minimisation, accès restreints, durée de conservation et base légale adaptée.

**Hypothèses sur les variables de la source principale :** 
1. Le type de transaction ("type") influence fortement le risque de fraude.

2. Les transactions frauduleuses présentent une distribution de montant
("amount") différente des transactions légitimes.

3. La relation entre le montant de la transaction et le solde disponible
avant transaction ("oldbalanceOrg") est associée au risque de fraude.

Sa nature synthétique motive la recherche d'une source complémentaire pour contextualiser la fraude réelle et discuter les limites du jeu de données.

## 5. Source complémentaire et sources écartées
### • Source complémentaire retenue

**[OSMP 2025 — Banque de France](https://www.banque-france.fr/fr/publications-et-statistiques/publications/rapport-de-lobservatoire-de-la-securite-des-moyens-de-paiement-2025).** Le PDF et le XLSX sont deux supports de la même source. Ils contextualisent la fraude réelle en France ; ils ne sont pas joints ligne par ligne à PaySim.

| Critère | Évaluation OSMP |
|---|---|
| Pertinence | Statistiques sur les usages et la fraude aux moyens de paiement. |
| Couverture | France, activité de 2025 ; champ exact à lire dans les notes de chaque tableau. |
| Fraîcheur | Publication et mise à jour de la page le 9 septembre 2026. |
| Granularité | Agrégats par période, indicateur et moyen de paiement ; pas de transactions individuelles. |
| Qualité | Publication institutionnelle avec notes et unités ; mise en page Excel distincte des observations. |
| Biais / limites | Contexte français et définitions différents du simulateur ; taux en valeur non comparables directement à un taux en nombre. |
| Réutilisation | Conditions Banque de France : intégrité, attribution et date de mise à jour, sous réserve de mentions particulières et de droits de tiers. |

Le classeur commence par un sommaire. Les tableaux T1/T3 des feuilles ci-dessous sont utiles au cadrage 2025. Leurs unités diffèrent : nombre en millions et montants en milliards pour T1 ; volumes en unités et montants en euros pour T3. La feuille interne `Annexe6-DIRCOM`, contenant notamment des éléments de 2022, n’est pas retenue.


In [26]:
fichier_osmp = racine / "docs/sources/Dossier_statistique_du_rapport_2025_de_l_OSMP_xlsx.xlsx"
with pd.ExcelFile(fichier_osmp) as classeur:
    print("Feuilles disponibles :", classeur.sheet_names)
    for feuille in ["Panorama des moyens de paiement", "Panorama de la fraude"]:
        tableau = pd.read_excel(classeur, sheet_name=feuille, header=None)
        print("Aperçu avec titres et unités :", feuille)
        display(tableau.iloc[:10].dropna(how="all").dropna(axis=1, how="all"))


Feuilles disponibles : ['SOMMAIRE', 'Panorama des moyens de paiement', 'Panorama de la fraude', 'Carte_émission', 'Carte_acceptation', 'Chèque', 'Virement', 'Prélèvement', 'Autres', 'Annexe6-DIRCOM']
Aperçu avec titres et unités : Panorama des moyens de paiement


,1,2,3,4,5,6,7,8,12
0,n PANORAMA DES MOYENS DE PAIEMENT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T1 Cartographie des moyens de paiement scriptu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Retour sommaire ~
2,"(nombre en millions, montant en milliards d’eu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Nombre de transactions,NaN,NaN,Montant des transactions,NaN,NaN,Montant moyen,NaN
4,NaN,2025,Variation\n2025/2024,Part,2025,Variation\n2025/2024,Part,NaN,NaN
5,Paiement carte a),22354.630469,6.537737,62.719878,875.862178,3.819125,2.415258,39.180347,NaN
6,dont sans contact,12653.712781,10.478865,35.502234,235.966705,15.701768,0.650696,18.648021,NaN
7,dont paiment par mobile,3413.403452,38.043271,9.576908,75.011613,33.905801,0.20685,21.975607,NaN
8,Chèque,673.777372,-14.040638,1.890402,330.61107,-15.709205,0.911686,490.682953,NaN
9,Virement,6362.184666,6.47848,17.850237,32415.942434,4.376176,89.389463,5095.096124,NaN


Aperçu avec titres et unités : Panorama de la fraude


,1,2,3,4,5,6,7,8,9
0,n PANORAMA DE LA FRAUDE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T3 Répartition de la fraude sur les moyens de ...,NaN,NaN,NaN,NaN,NaN,NaN,Retour sommaire ~,NaN
2,(valeur et montant moyen en euros ; volume en ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Volume,NaN,NaN,Valeur,NaN,NaN,Taux de fraude,Montant
4,NaN,2025,Variation\n2025/2024,Part,2025,Variation\n2025/2024,Part,2025,moyen
5,Paiement carte a),6582317,-10.001642,91.514338,432455222,0,34.852811,0.049375,65.699543
6,dont sans contact,700825,-6.171721,9.743611,21142482,-6.648071,1.703933,0.00896,30.167991
7,dont par mobile,106252,-17.667297,1.477228,6879848,-23.753756,0.554467,0.009172,64.750292
8,Chèque (nouvelle approche) b),158405,-12.026058,2.202314,228944293.24,-15.952588,18.45128,0.069249,1445.309764
9,Chèque (ancienne approche),198019,-11.325115,2.75307,387099402.87,-17.186614,31.197455,0.117086,1954.859902


**RGPD de la source complémentaire retenue:**

Le rapport OSMP utilisé comme source complémentaire contient des statistiques agrégées et ne contient pas, dans notre usage du projet, de données permettant d’identifier directement des personnes.

Le risque RGPD lié à cette source est donc faible.

### • Sources non retenues

MoMTSim a été examiné comme source alternative de transactions de mobile. Elle a été écarté car il s’agit également d’un jeu synthétique et il n’apporte pas de contexte réel supplémentaire par rapport à PaySim, qui est imposé dans le sujet.

Le communiqué OSMP du premier semestre 2025 a été examiné mais non retenu au profit du rapport annuel 2025, plus complet pour le cadrage du projet.


In [27]:
sources_ecartees = pd.read_csv(racine / "docs/sources_ecartees.csv", sep=";")
with pd.option_context("display.max_colwidth", 120):
    display(sources_ecartees)


,source,url,decision,justification
0,Communiqué OSMP S1 2025,https://www.banque-france.fr/fr/actualites/communique-de-presse-de-lobservatoire-de-la-securite-des-moyens-de-paiement,Non retenue,"Couverture semestrielle, moins adaptée au cadrage annuel 2025 retenu."
1,MoMTSim — Mendeley Data V2,https://data.mendeley.com/datasets/zhj366m53p/2,Non retenue,"PaySim est imposé ; autre simulateur synthétique, sans contexte réel supplémentaire pour M1."


## 6. Traçabilité et conditions de réutilisation

Le [journal](journal_lineage.csv) identifie les fichiers du rendu et les deux fichiers MoMTSim examinés.

Le fichier PaySim utilisé est identifié par son SHA-256. La sortie enregistrée de l’extraction indique la version Kaggle 2, mais le téléchargement actuel ne la fige pas explicitement : l’empreinte est la référence du fichier de ce rendu.

PaySim : CC BY-SA 4.0 consignée. OSMP : [conditions de réutilisation Banque de France](https://www.banque-france.fr/fr/infos-legales). 
MoMTSim : CC BY 4.0 indiquée sur la [fiche Mendeley V2](https://data.mendeley.com/datasets/zhj366m53p/2). Les références doivent être citées et toute redistribution respecter leurs conditions.

### Accès aux fichiers bruts

Le CSV PaySim et les fichiers bruts MoMTSim ne sont volontairement pas versionnés dans Git en raison de leur taille. Pour exécuter la documentation, lancer d’abord `01_extraction_paysim.ipynb` : il télécharge PaySim depuis [Kaggle](https://www.kaggle.com/datasets/ealaxi/paysim1) dans le cache KaggleHub. Ce notebook réutilise le CSV en cache avec un chemin relatif à la racine du projet ; une copie déjà présente dans `docs/sources/PS_20174392719_1491204439457_log-2.csv` reste prioritaire. La version observée et le SHA-256 de référence figurent dans le [journal de lineage](journal_lineage.csv) ; vérifier l’empreinte après téléchargement.

MoMTSim reste une source écartée, consultable sur [Mendeley Data V2](https://data.mendeley.com/datasets/zhj366m53p/2). Ses fichiers ne sont pas nécessaires aux notebooks et doivent rester hors du dépôt. Le PDF et le XLSX OSMP sont inclus comme supports de la source complémentaire agrégée, sans jointure transactionnelle avec PaySim.


In [28]:
lineage = pd.read_csv(racine / "docs/journal_lineage.csv", sep=";").fillna("")
display(lineage[["source", "statut", "usage", "date_recuperation", "version"]])
with pd.option_context("display.max_colwidth", 100):
    display(lineage[["source", "fichier", "sha256", "transformations"]])


,source,statut,usage,date_recuperation,version
0,PaySim,Retenue,Source principale — analyse transactionnelle,02/10/2026,2 observée dans la sortie initiale ; non figée...
1,Banque de France / Observatoire de la sécurité...,Retenue,Source complémentaire sélectionnée — contexte ...,03/10/2026,Rapport 2025
2,Banque de France / OSMP 2025 - dossier statist...,Retenue,Source complémentaire sélectionnée — contexte ...,03/10/2026,Dossier statistique 2025
3,MoMTSim / Mendeley Data,Écartée — fichiers hors livrable,"Alternative examinée, non utilisée",Non documentée — à confirmer,2 ; DOI 10.17632/zhj366m53p.2
4,MoMTSim / Mendeley Data,Écartée — fichiers hors livrable,"Alternative examinée, non utilisée",Non documentée — à confirmer,2 ; DOI 10.17632/zhj366m53p.2


,source,fichier,sha256,transformations
0,PaySim,docs/sources/PS_20174392719_1491204439457_log-2.csv,16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b,"Brut inchangé. Contrôles shape, types, valeurs manquantes, doublons et cible. df_model retire ne..."
1,Banque de France / Observatoire de la sécurité des moyens de paiement,docs/sources/Rapport_2025_de_l_OSMP_pdf.pdf,b9cce05264798f7a3fd24e7d57ff8cd5cce7d533701abaca40b6d688366f8f96,Consultation du PDF et aperçu des tableaux T1/T3 du XLSX ; aucune jointure ni modification des s...
2,Banque de France / OSMP 2025 - dossier statistique,docs/sources/Dossier_statistique_du_rapport_2025_de_l_OSMP_xlsx.xlsx,1afb95403c65842310bc430d3fc523dd3e965ad9b3a48441e2753e95e82cadde,Consultation du PDF et aperçu des tableaux T1/T3 du XLSX ; aucune jointure ni modification des s...
3,MoMTSim / Mendeley Data,Hors livrable : Downloads/MoMTSim_20240722202413_1000_dataset.csv,99fd07c3a9d3c4bd6d3462240058ca19d0d9e9284683f78bf77542ff7fcc05e7,Contrôles historiques sur les CSV ; aucune fusion ni modification. Copies du livrable retirées ;...
4,MoMTSim / Mendeley Data,Hors livrable : Downloads/synthetic_mobile_money_transaction_dataset.csv,da951eb95735da96271740a3e66b676b342d3831ce3111cd19dbfa020d3bd0a7,Contrôles historiques sur les CSV ; aucune fusion ni modification. Copies du livrable retirées ;...


La cellule suivante vérifie les empreintes des trois fichiers conservés, sans les modifier.


In [29]:
import hashlib

for _, ligne in lineage[lineage.statut == "Retenue"].iterrows():
    fichier = (fichier_paysim() if ligne.fichier == "docs/sources/PS_20174392719_1491204439457_log-2.csv"
               else racine / ligne.fichier)
    empreinte = hashlib.sha256()
    with fichier.open("rb") as flux:
        for bloc in iter(lambda: flux.read(1024 * 1024), b""):
            empreinte.update(bloc)
    assert empreinte.hexdigest() == ligne.sha256, f"Empreinte différente : {fichier.name}"
    print("Empreinte conforme :", fichier.name)


Empreinte conforme : PS_20174392719_1491204439457_log-2.csv
Empreinte conforme : Rapport_2025_de_l_OSMP_pdf.pdf
Empreinte conforme : Dossier_statistique_du_rapport_2025_de_l_OSMP_xlsx.xlsx


## 7. Transmission à l’équipe

- Cible : `isFraud`; unité : une transaction ; décision avant exécution.
- Fraudes très minoritaires : compléter l’accuracy par des métriques adaptées.
- Exclure les soldes après transaction et la sortie de règle `isFlaggedFraud`.
- Les identifiants ont une forte cardinalité ; leur utilisation brute n’est pas recommandée.
- Les données sont synthétiques ; ne pas généraliser directement au réel.
- OSMP apporte du contexte agrégé, sans jointure transactionnelle.
- Le sujet recommande un échantillon stratifié pour l’EDA, à documenter.
- Conserver les empreintes, les versions et un protocole reproductible.

Le [Data Canvas](data_canvas.md) rassemble le cadrage métier et les responsabilités liées aux données.
